In [ ]:
#===================imports=======================
import cv2  # OpenCV for video processing
import mediapipe as mp # MediaPipe for hand tracking
import numpy as np # NumPy for numerical operations
import time # Time module for measuring FPS

from mediapipe.tasks import python  # Importing the Python API for MediaPipe tasks
from mediapipe.tasks.python import vision # Importing the vision module from MediaPipe tasks
import math # Math module for calculating angles
# ================== MODEL PATHS ==================
POSE_MODEL_PATH = "pose_landmarker_full.task" # Path to the pose landmarker model
HAND_MODEL_PATH = "hand_landmarker.task" # Path to the hand landmarker model
# ================== BASE OPTIONS ==================
BaseOptions = python.BaseOptions # Base options for configuring the MediaPipe tasks
VisionRunningMode = vision.RunningMode # Running mode for the vision tasks (e.g., video, image, live stream)
# ================== POSE SETUP ==================
PoseLandmarker = vision.PoseLandmarker # PoseLandmarker class for pose estimation
PoseLandmarkerOptions = vision.PoseLandmarkerOptions # Options for configuring the PoseLandmarker

pose_options = PoseLandmarkerOptions( # Configuring the options for the PoseLandmarker
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH), # Setting the model path for the pose landmarker
    running_mode=VisionRunningMode.VIDEO, # Setting the running mode to video for frame-by-frame processing
    num_poses=1 # Setting the number of poses to detect (1 in this case)
)
# ================== HAND SETUP ==================
HandLandmarker = vision.HandLandmarker # HandLandmarker class for hand tracking
HandLandmarkerOptions = vision.HandLandmarkerOptions # Options for configuring the HandLandmarker

hand_options = HandLandmarkerOptions( # Configuring the options for the HandLandmarker
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH), # Setting the model path for the hand landmarker
    running_mode=VisionRunningMode.VIDEO, # Setting the running mode to video for frame-by-frame processing
    num_hands=2 # Setting the number of hands to detect (2 in this case)
)
# ================== ANGLE CALCULATION ==================
def calculate_angle(a, b, c ,h , w):# Function to calculate the angle between three points (a, b, c)
    a = np.array([a.x * w, a.y * h, a.z * w])
    b = np.array([b.x * w, b.y * h, b.z * w])
    c = np.array([c.x * w, c.y * h, c.z * w])

    # Vectors from b
    ba = a - b
    bc = c - b

    # Angle calculation
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)  # prevent numerical errors
    angle = np.arccos(cosine_angle)

    return np.degrees(angle)
def calculate_elbow_angle(shoulder, elbow, wrist, w, h):

    shoulder = np.array([shoulder.x * w, shoulder.y * h])
    elbow = np.array([elbow.x * w, elbow.y * h])
    wrist = np.array([wrist.x * w, wrist.y * h])

    upper_arm = shoulder - elbow
    forearm = wrist - elbow

    cosine_angle = np.dot(upper_arm, forearm) / (
        np.linalg.norm(upper_arm) * np.linalg.norm(forearm)
    )

    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    angle = np.arccos(cosine_angle)

    return np.degrees(angle)

def draw_connections(frame, landmarks, connections, w, h):
    """
    Draws bold light blue lines for bones and red circles for joints.
    
    Parameters:
    - frame: the image to draw on
    - landmarks: list of landmarks
    - connections: list of tuples (start_idx, end_idx)
    - w, h: image width and height
    """
    line_color = (255, 200, 0)  # Light blue in BGR (OpenCV uses BGR)
    line_thickness = 4           # Bold lines

    # Draw connections (bones)
    for start_idx, end_idx in connections:
        start = landmarks[start_idx]
        end = landmarks[end_idx]

        x1, y1 = int(start.x * w), int(start.y * h)
        x2, y2 = int(end.x * w), int(end.y * h)

        cv2.line(frame, (x1, y1), (x2, y2), line_color, line_thickness)

    # Draw joints as red circles
    for lm in landmarks:
        x, y = int(lm.x * w), int(lm.y * h)
        cv2.circle(frame, (x, y), 6, (0, 0, 255), -1)  # Red filled circle
        # ================== POSE CONNECTIONS ==================
POSE_CONNECTIONS = [
    (11, 12),             # Shoulders
    (11, 13), (13, 15),   # Left arm
    (12, 14), (14, 16),   # Right arm
    (11, 23), (12, 24)    # Torso
]

# ================== HAND CONNECTIONS ==================
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),       # Thumb
    (0,5),(5,6),(6,7),(7,8),       # Index finger
    (0,9),(9,10),(10,11),(11,12),  # Middle finger
    (0,13),(13,14),(14,15),(15,16),# Ring finger
    (0,17),(17,18),(18,19),(19,20) # Little finger
]
def process_left_side(frame, landmarks, w, h):
    """
    Draws left side pose connections and calculates shoulder/elbow angles.
    Text is displayed near the top-left inside the frame.
    """
    left_connections = [
        (11, 13), (13, 15),  # Left arm
        (11, 23)              # Left torso
    ]
    draw_connections(frame, landmarks, left_connections, w, h)

    # Shoulder (left) angle
    shoulder_angle = calculate_angle(
        landmarks[23],  # hip
        landmarks[11],  # shoulder
        landmarks[13],  # elbow
        w, h
    )
    cv2.putText(frame, f"L Shoulder: {int(shoulder_angle)}°",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255,255,0), 2)
    
    # Elbow (left) angle
    elbow_angle = calculate_elbow_angle(
        landmarks[11],  # shoulder
        landmarks[13],  # elbow
        landmarks[15],  # wrist
        w, h
    )
    cv2.putText(frame, f"L Elbow: {int(elbow_angle)}°", 
                (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255,255,0), 2)
    
    return shoulder_angle, elbow_angle


def process_right_side(frame, landmarks, w, h):
    """
    Draws right side pose connections and calculates shoulder/elbow angles.
    Text is displayed near the top-right inside the frame.
    """
    right_connections = [
        (12, 14), (14, 16),  # Right arm
        (12, 24)              # Right torso
    ]
    draw_connections(frame, landmarks, right_connections, w, h)

    # Shoulder (right) angle
    shoulder_angle = calculate_angle(
        landmarks[24],  # hip
        landmarks[12],  # shoulder
        landmarks[14],  # elbow
        w, h
    )
    cv2.putText(frame, f"R Shoulder: {int(shoulder_angle)}°",
                (w - 220, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255,255,0), 2)
    
    # Elbow (right) angle
    elbow_angle = calculate_elbow_angle(
        landmarks[12],  # shoulder
        landmarks[14],  # elbow
        landmarks[16],  # wrist
        w , h
    )
    cv2.putText(frame, f"R Elbow: {int(elbow_angle)}°",
                (w - 220, 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255,255,0), 2)
    
    return shoulder_angle, elbow_angle
# ================== CAMERA ==================
cap = cv2.VideoCapture(0)
start_time = time.time()

window_name = "Physio Assessment - Hand + Arm"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

baseline_open_angle = None
side_selected = None  # User must press 'L' or 'R'

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    while True:

        # Stop if window closed
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = int((time.time() - start_time) * 1000)

        # ================== POSE DETECTION ==================
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if pose_result.pose_landmarks:
            landmarks = pose_result.pose_landmarks[0]

            if side_selected == 'L':
                left_shoulder, left_elbow = process_left_side(frame, landmarks, w, h)

            elif side_selected == 'R':
                right_shoulder, right_elbow = process_right_side(frame, landmarks, w, h)

            else:
                cv2.putText(frame, "Press 'L' for Left side, 'R' for Right side",
                            (30, 30),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.8, (0, 255, 255), 2)

        # ================== HAND DETECTION ==================
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if hand_result.hand_landmarks and hand_result.handedness:

            for i in range(len(hand_result.hand_landmarks)):

                hand_landmarks = hand_result.hand_landmarks[i]
                hand_label = hand_result.handedness[i][0].category_name.lower()

                # Ignore the non-selected hand
                if side_selected == 'L' and hand_label != "left":
                    continue

                if side_selected == 'R' and hand_label != "right":
                    continue

                # Draw chosen hand
                draw_connections(frame, hand_landmarks, HAND_CONNECTIONS, w, h)

                # MCP Angles
                index_angle = calculate_angle(hand_landmarks[0], hand_landmarks[5], hand_landmarks[8], w, h)
                middle_angle = calculate_angle(hand_landmarks[0], hand_landmarks[9], hand_landmarks[12], w, h)
                ring_angle = calculate_angle(hand_landmarks[0], hand_landmarks[13], hand_landmarks[16], w, h)
                pinky_angle = calculate_angle(hand_landmarks[0], hand_landmarks[17], hand_landmarks[20], w, h)

                avg_angle = (index_angle + middle_angle + ring_angle + pinky_angle) / 4

                # Convert to percentage
                open_percentage = int(((avg_angle - 60) / (180 - 60)) * 100)
                open_percentage = max(0, min(100, open_percentage))
                closed_percentage = 100 - open_percentage

                text_x = 30 if side_selected == 'L' else w - 200

                cv2.putText(frame, f"Open: {open_percentage}%",
                            (text_x, 120),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,255,0), 2)

                cv2.putText(frame, f"Closed: {closed_percentage}%",
                            (text_x, 160),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,0,255), 2)

        # ================== SHOW FRAME ==================
        cv2.imshow(window_name, frame)

        # ================== KEY INPUT ==================
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

        elif key == ord('l') or key == ord('L'):
            side_selected = 'L'

        elif key == ord('r') or key == ord('R'):
            side_selected = 'R'


# ================== CLEANUP ==================
cap.release()
cv2.destroyAllWindows()


In [1]:
#===================imports=======================
import cv2
import mediapipe as mp
import numpy as np
import time
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import math

# ================== MODEL PATHS ==================
POSE_MODEL_PATH = "pose_landmarker_full.task"
HAND_MODEL_PATH = "hand_landmarker.task"

# ================== BASE OPTIONS ==================
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

# ================== POSE SETUP ==================
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

# ================== HAND SETUP ==================
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

# ================== ANGLE SMOOTHING ==================
prev_left_elbow = None
prev_right_elbow = None
prev_left_shoulder = None
prev_right_shoulder = None

SMOOTHING = 0.7

def smooth_angle(previous, current, alpha=0.7):
    if previous is None:
        return current
    return alpha * previous + (1 - alpha) * current

# ================== ANGLE CALCULATION ==================
def calculate_angle(a, b, c, w, h):

    a = np.array([a.x * w, a.y * h])
    b = np.array([b.x * w, b.y * h])
    c = np.array([c.x * w, c.y * h])

    ba = a - b
    bc = c - b

    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

def calculate_elbow_angle(shoulder, elbow, wrist, w, h):

    shoulder = np.array([shoulder.x * w, shoulder.y * h])
    elbow = np.array([elbow.x * w, elbow.y * h])
    wrist = np.array([wrist.x * w, wrist.y * h])

    upper_arm = shoulder - elbow
    forearm = wrist - elbow

    cosine_angle = np.dot(upper_arm, forearm) / (
        np.linalg.norm(upper_arm) * np.linalg.norm(forearm)
    )

    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

# ================== DRAW CONNECTIONS ==================
def draw_connections(frame, landmarks, connections, w, h):

    line_color = (255, 200, 0)
    line_thickness = 4

    for start_idx, end_idx in connections:

        start = landmarks[start_idx]
        end = landmarks[end_idx]

        x1, y1 = int(start.x * w), int(start.y * h)
        x2, y2 = int(end.x * w), int(end.y * h)

        cv2.line(frame, (x1, y1), (x2, y2), line_color, line_thickness)

    for lm in landmarks:
        x, y = int(lm.x * w), int(lm.y * h)
        cv2.circle(frame, (x, y), 6, (0, 0, 255), -1)

# ================== HAND CONNECTIONS ==================
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20)
]

# ================== LEFT SIDE ==================
def process_left_side(frame, landmarks, w, h):

    global prev_left_elbow, prev_left_shoulder

    left_connections = [(11,13),(13,15),(11,23)]
    draw_connections(frame, landmarks, left_connections, w, h)

    shoulder_angle = calculate_angle(
        landmarks[23], landmarks[11], landmarks[13], w, h
    )

    elbow_angle = calculate_elbow_angle(
        landmarks[11], landmarks[13], landmarks[15], w, h
    )

    shoulder_angle = smooth_angle(prev_left_shoulder, shoulder_angle, SMOOTHING)
    elbow_angle = smooth_angle(prev_left_elbow, elbow_angle, SMOOTHING)

    prev_left_shoulder = shoulder_angle
    prev_left_elbow = elbow_angle

    cv2.putText(frame, f"L Shoulder: {int(shoulder_angle)}°",
                (20,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0),2)

    cv2.putText(frame, f"L Elbow: {int(elbow_angle)}°",
                (20,70), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0),2)

# ================== RIGHT SIDE ==================
def process_right_side(frame, landmarks, w, h):

    global prev_right_elbow, prev_right_shoulder

    right_connections = [(12,14),(14,16),(12,24)]
    draw_connections(frame, landmarks, right_connections, w, h)

    shoulder_angle = calculate_angle(
        landmarks[24], landmarks[12], landmarks[14], w, h
    )

    elbow_angle = calculate_elbow_angle(
        landmarks[12], landmarks[14], landmarks[16], w, h
    )

    shoulder_angle = smooth_angle(prev_right_shoulder, shoulder_angle, SMOOTHING)
    elbow_angle = smooth_angle(prev_right_elbow, elbow_angle, SMOOTHING)

    prev_right_shoulder = shoulder_angle
    prev_right_elbow = elbow_angle

    cv2.putText(frame, f"R Shoulder: {int(shoulder_angle)}°",
                (w-220,40), cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,0),2)

    cv2.putText(frame, f"R Elbow: {int(elbow_angle)}°",
                (w-220,70), cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,0),2)

# ================== CAMERA ==================
cap = cv2.VideoCapture(0)
start_time = time.time()

window_name = "Physio Assessment - Hand + Arm"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

side_selected = None

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    while True:

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = int((time.time() - start_time) * 1000)

        # ================== POSE DETECTION ==================
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if pose_result.pose_landmarks:

            landmarks = pose_result.pose_landmarks[0]

            if side_selected == 'L':
                process_left_side(frame, landmarks, w, h)

            elif side_selected == 'R':
                process_right_side(frame, landmarks, w, h)

            else:
                cv2.putText(frame,
                            "Press L for Left side, R for Right side",
                            (30,30),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.8,(0,255,255),2)

        # ================== HAND DETECTION ==================
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if hand_result.hand_landmarks and hand_result.handedness:

            for i in range(len(hand_result.hand_landmarks)):

                hand_landmarks = hand_result.hand_landmarks[i]
                hand_label = hand_result.handedness[i][0].category_name.lower()

                if side_selected == 'L' and hand_label != "left":
                    continue

                if side_selected == 'R' and hand_label != "right":
                    continue

                draw_connections(frame, hand_landmarks, HAND_CONNECTIONS, w, h)

                index_angle = calculate_angle(hand_landmarks[0], hand_landmarks[5], hand_landmarks[8], w, h)
                middle_angle = calculate_angle(hand_landmarks[0], hand_landmarks[9], hand_landmarks[12], w, h)
                ring_angle = calculate_angle(hand_landmarks[0], hand_landmarks[13], hand_landmarks[16], w, h)
                pinky_angle = calculate_angle(hand_landmarks[0], hand_landmarks[17], hand_landmarks[20], w, h)

                avg_angle = (index_angle + middle_angle + ring_angle + pinky_angle) / 4

                open_percentage = int(((avg_angle - 60) / (180 - 60)) * 100)
                open_percentage = max(0, min(100, open_percentage))
                closed_percentage = 100 - open_percentage

                text_x = 30 if side_selected == 'L' else w-200

                cv2.putText(frame, f"Open: {open_percentage}%",
                            (text_x,120),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9,(0,255,0),2)

                cv2.putText(frame, f"Closed: {closed_percentage}%",
                            (text_x,160),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9,(0,0,255),2)

        # ================== SHOW FRAME ==================
        cv2.imshow(window_name, frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break
        elif key == ord('l') or key == ord('L'):
            side_selected = 'L'
        elif key == ord('r') or key == ord('R'):
            side_selected = 'R'

cap.release()
cv2.destroyAllWindows()